In [9]:
nions = 13

new_file_suffix = "v1"  # new filename will be  <date>_am_<nions>_<new_file_suffix>.json
prev_solution_file = "../out/20260403_am_13_symmetric_src.json"  # previous solution to tweak all gates from

params_file = "../system_params/072125_goldparams_13ions.json"

"../system_params/072125_goldparams_13ions.json"

In [10]:
using GoldGates
using MSSim: Optimizers as Opts, SegSeq as SS, SymLinear as SL, Sequence as Seq, Utils as U
using NLopt
using Statistics
using PyPlot
using JSON
using Dates

include("../src/utils.jl")
ENV["MPLBACKEND"] = "module://matplotlib_inline.backend_inline"

sysparams = open(params_file) do io
    read(io, GoldGates.SystemParams; format=:json)
end

date = Dates.format(Dates.now(), "yyyymmdd")

# Load all ion pair keys from previous solution and derive nseg
prev_data = open(prev_solution_file) do io
    JSON.parse(io)
end
all_ion_pair_keys = collect(keys(prev_data["XX"]))
nseg = Int(prev_data["XX"][all_ion_pair_keys[1]]["nsteps"])

out_file = "../out/$(date)_am_$(nions)_$(new_file_suffix).json"

println("Found $(length(all_ion_pair_keys)) ion pairs to process, nseg=$nseg")
println(all_ion_pair_keys)

Found 20 ion pairs to process, nseg=30
["2,-4", "3,-4", "-1,-3", "-1,-2", "1,-2", "0,-4", "3,-3", "0,-1", "-1,-4", "1,-1", "0,-3", "2,-2", "0,-2", "-2,-3", "2,-3", "-2,-4", "1,-3", "-3,-4", "4,-4", "1,-4"]


In [11]:
all_failures = Dict{String, Vector{String}}()

for ion_pair_key in all_ion_pair_keys
    parts = split(ion_pair_key, ",")
    ion1 = parse(Int, parts[1])
    ion2 = parse(Int, parts[2])

    println("=== Processing ions ($ion1, $ion2) ===")

    prev_meta = load_solution_metadata(prev_solution_file, ion_pair_key)
    if prev_meta === nothing
        @warn "No metadata for $ion_pair_key, skipping"
        continue
    end
    prev_params = get(prev_meta, "opt_params", nothing)
    if prev_params === nothing
        @warn "No opt_params for $ion_pair_key, skipping"
        continue
    end

    pitime = prev_meta["pitime"]
    τmin = prev_meta["τmin"]
    τmax = prev_meta["τmax"]
    maxtime = prev_meta["maxtime"]
    min_mode_index = Int(prev_meta["min_mode_index"])
    max_mode_index = Int(prev_meta["max_mode_index"])

    modes = setup_modes(sysparams, ion1, ion2, nions)
    nlmodel = setup_model(nseg, modes)
    opt, tracker = setup_optimizer(nlmodel, sysparams; pitime, τmin, τmax, maxtime, min_mode_index, max_mode_index)

    best_params, best_obj = run_optimization!(opt, tracker, nlmodel;
        initial_params=prev_params, perturbation=0.05)

    opt_raw_params, metadata = get_metadata_and_plot(nlmodel, best_params, nseg, sysparams, modes;
        ion1, ion2, pitime, τmin, τmax, maxtime, min_mode_index, max_mode_index, plot=false)

    failures = verify_solution(metadata)
    if !isempty(failures)
        all_failures[ion_pair_key] = failures
    end

    save_am_solution(out_file, opt_raw_params, metadata, sysparams, ion1, ion2)
end

println("
=== Done processing all $(length(all_ion_pair_keys)) ion pairs ===")


=== Processing ions (2, -4) ===
(obj = 6.685331565657815e-7, dis = 3.553772308069753e-10, disδ = 1.7142603485482415e-5, area = 1.5707963222556962, areaε = -4.5392003489297394e-9, areaδ = 7.03797011571155, total_t = 303.9043955977873, Ωmax = 0.0)
(obj = 6.685331554653203e-7, dis = 3.553752747898098e-10, disδ = 1.714260304334192e-5, area = 1.5707963231012345, areaε = -3.6936620428917877e-9, areaδ = 7.037970213730931, total_t = 303.9043954840826, Ωmax = 0.0)
(obj = 6.685331551832411e-7, dis = 3.5537415414497053e-10, disδ = 1.714260344807293e-5, area = 1.570796322586133, areaε = -4.2087635598875295e-9, areaδ = 7.037970219888813, total_t = 303.9043955203962, Ωmax = 0.0)
  0.846751 seconds (1.13 M allocations: 56.526 MiB, 2.01% gc time, 44.06% compilation time: 100% of which was recompilation)
Saved solution for ions (2, -4) to ../out/20260403_am_13_v1.json


┌ Warning: carrier_pi_time_required = 2.58 (>= 2.7)
└ @ Main /Users/vivz/Documents/git/GoldGates.jl/src/utils.jl:250


=== Processing ions (3, -4) ===
(obj = 4.968927125423028e-7, dis = 4.6253619284831505e-11, disδ = 1.304972655849752e-5, area = -1.5707963238181646, areaε = -2.976731972026414e-9, areaδ = 6.0511501218760895, total_t = 426.17067375998477, Ωmax = 0.0)
(obj = 4.7673987242440917e-7, dis = 1.995921685571357e-11, disδ = 5.278010783477746e-6, area = -1.5707963234262634, areaε = -3.368633150202527e-9, areaδ = 6.51045288263869, total_t = 425.92353043713234, Ωmax = 0.0)
(obj = 4.7107728293442694e-7, dis = 1.098909212536881e-11, disδ = 3.467379618074993e-6, area = -1.570796312447281, areaε = -1.434761554008901e-8, areaδ = 6.605668177464283, total_t = 425.87394718883525, Ωmax = 0.0)
(obj = 2.3485946978637552e-7, dis = 2.1919876810595045e-10, disδ = 2.080194844737604e-5, area = -1.5707963291836071, areaε = 2.388710562684082e-9, areaδ = 1.6044934060784282, total_t = 428.813814384947, Ωmax = 0.0)
(obj = 1.2821449082472267e-7, dis = 8.844209500333072e-11, disδ = 6.386362840413597e-6, area = -1.57079632

┌ Warning: carrier_pi_time_required = 2.398 (>= 2.7)
└ @ Main /Users/vivz/Documents/git/GoldGates.jl/src/utils.jl:250


 = 4.3886815718040483e-7, dis = 2.4640980995977194e-10, disδ = 4.22018542902849e-5, area = -1.5707963252590156, areaε = -1.535880977954207e-9, areaδ = 1.249702564287356, total_t = 390.5946049422988, Ωmax = 0.0)
(obj = 4.374708504993642e-7, dis = 2.4811760176169073e-10, disδ = 4.202056967205855e-5, area = -1.5707963251851809, areaε = -1.6097156940730883e-9, areaδ = 1.2658817287113275, total_t = 390.58178736507557, Ωmax = 0.0)
(obj = 2.3580743402207103e-7, dis = 1.8875675134901828e-10, disδ = 2.3272116429306216e-5, area = -1.570796325870742, areaε = -9.241545306792887e-10, areaδ = 0.4628699479181642, total_t = 391.20979105557484, Ωmax = 0.0)
  1.722191 seconds (187.63 k allocations: 13.555 MiB, 0.05% gc time)
All verification checks passed.
Saved solution for ions (1, -2) to ../out/20260403_am_13_v1.json
=== Processing ions (0, -4) ===
(obj = 2.9863926428061413e-6, dis = 1.2089500372341079e-10, disδ = 1.162319293931277e-5, area = -1.5707963081569523, areaε = -1.8637944299015885e-8, areaδ

...

 = 0.0007488290441451349, dis = 8.729846206630362e-7, disδ = 0.07389716582545716, area = -1.5707939257720807, areaε = -2.401022815901044e-6, areaδ = -23.434773940005883, total_t = 350.7283688417188, Ωmax = 0.0)
(obj = 1.675564119429495e-6, dis = 1.7480392305265044e-10, disδ = 4.232403734513999e-5, area = -1.5707963185276745, areaε = -8.267222018076836e-9, areaδ = -11.186821351608934, total_t = 389.6388336038735, Ωmax = 0.0)
(obj = 1.673760360430828e-6, dis = 1.7814140819621172e-10, disδ = 4.23476727395241e-5, area = -1.570796319049674, areaε = -7.745222463384493e-9, areaδ = -11.177624613466667, total_t = 389.63226664769087, Ωmax = 0.0)
(obj = 1.6155816148143983e-6, dis = 1.5881606013092568e-10, disδ = 6.0648227679021374e-5, area = -1.570796318397633, areaε = -8.397263551174206e-9, areaδ = -10.041440388072452, total_t = 388.82424352962335, Ωmax = 0.0)
  1.474279 seconds (175.10 k allocations: 12.616 MiB, 0.10% gc time)
All verification checks passed.
Saved solution for ions (2, -3) to .

┌ Warning: carrier_pi_time_required = 2.576 (>= 2.7)
└ @ Main /Users/vivz/Documents/git/GoldGates.jl/src/utils.jl:250


Saved solution for ions (-3, -4) to ../out/20260403_am_13_v1.json
=== Processing ions (4, -4) ===
(obj = 1.3999500294756304e-7, dis = 2.682558619818934e-10, disδ = 1.2945990794787063e-5, area = -1.570796326320165, areaε = -4.747315873743219e-10, areaδ = 0.9588438698373125, total_t = 303.904395572331, Ωmax = 0.0)
(obj

┌ Warning: displacement_at_+1kHz = 0.05845 (<= 0.05)
└ @ Main /Users/vivz/Documents/git/GoldGates.jl/src/utils.jl:250
┌ Warning: displacement_at_-1kHz = 0.05246 (<= 0.05)
└ @ Main /Users/vivz/Documents/git/GoldGates.jl/src/utils.jl:250


 = 1.3999500225322676e-7, dis = 2.682562980233085e-10, disδ = 1.2945990500332884e-5, area = -1.5707963263235545, areaε = -4.713420764801413e-10, areaδ = 0.9588438735041818, total_t = 303.90439559023133, Ωmax = 0.0)
  0.493557 seconds (62.50 k allocations: 4.503 MiB)
All verification checks passed.
Saved solution for ions (4, -4) to ../out/20260403_am_13_v1.json
=== Processing ions (1, -4) ===
(obj = 1.480044569114887e-5, dis = 1.7740826981219874e-8, disδ = 0.0014039717464502594, area = 1.570796277611167, areaε = -4.918372953355288e-8, areaδ = -8.197706080582865, total_t = 397.0090819573641, Ωmax = 5.170345787100527e-6)
(obj = 3.5874954629460738e-6, dis = 4.224416367285083e-10, disδ = 5.95343355979309e-5, area = 1.5707963064830641, areaε = -2.0311832438224542e-8, areaδ = 17.291731716418887, total_t = 434.445243965576, Ωmax = 0.0)
(obj = 4.4187381042440414e-7, dis = 1.6679261812812107e-10, disδ = 2.888717957110554e-5, area = 1.5707963259777935, areaε = -8.171030518866473e-10, areaδ = 3.9

In [12]:
println("
===== VERIFICATION REPORT =====")
println("Total pairs processed: $(length(all_ion_pair_keys))")
n_failed = length(all_failures)
n_passed = length(all_ion_pair_keys) - n_failed
println("Passed: $n_passed / $(length(all_ion_pair_keys))")

if isempty(all_failures)
    println("All gates passed verification!")
else
    println("
Failed gates ($n_failed):")
    for (pair_key, failures) in sort(collect(all_failures))
        println("  Ion pair $pair_key:")
        for f in failures
            println("    - $f")
        end
    end
end


===== VERIFICATION REPORT =====
Total pairs processed: 20
Passed: 12 / 20

Failed gates (8):
  Ion pair -1,-2:
    - carrier_pi_time_required = 2.398 (>= 2.7)
  Ion pair -1,-4:
    - carrier_pi_time_required = 2.506 (>= 2.7)
  Ion pair -2,-3:
    - carrier_pi_time_required = 2.673 (>= 2.7)
  Ion pair -3,-4:
    - displacement_at_+1kHz = 0.05845 (<= 0.05)
    - displacement_at_-1kHz = 0.05246 (<= 0.05)
  Ion pair 0,-3:
    - carrier_pi_time_required = 2.652 (>= 2.7)
  Ion pair 0,-4:
    - carrier_pi_time_required = 2.576 (>= 2.7)
  Ion pair 1,-3:
    - carrier_pi_time_required = 2.576 (>= 2.7)
  Ion pair 2,-4:
    - carrier_pi_time_required = 2.58 (>= 2.7)
